<style>table { margin-left: 0 !important; } td, th { text-align: left !important; }</style>

# SageAgent V4

AI coding assistant for SageMaker notebooks. 25+ tools, 16 security layers, prompt caching, sub-agent coordination, skills system. v4.3.2.

**Setup:** Run cells 1-3 in order. Cell 1 installs packages (once). Cell 2 shows config widgets. Cell 3 launches the agent.

**Core files:** `sagemaker_agent.py` (8,699 lines — the entire agent) + this notebook + `memory.md` (auto-populated) + `skills/` (optional).

**Docs:** See `USER_GUIDE.md` for full documentation. Cell 4 below has a quick reference covering tools, skills, memory, security, caching, and commands.

## How It Works (30-second version)

1. You type a message → sent to Claude via AWS Bedrock
2. Claude picks the right tool (read file, run bash, edit code, etc.) → executes it locally
3. Result sent back to Claude → repeats until done (up to 60 turns)
4. **Skills** = instruction files that make the agent follow checklists (review, verify, coding-standards)
5. **Memory** = persistent facts the agent learns about you across sessions (`memory.md`)
6. **Security** = 16 layers validate every tool call before execution

## Recommended Workflow

```
/skill use coding-standards          ← turn on at start (stays active all session)
"write a function to parse CSV"      ← agent follows standards while coding
/verify                              ← after done, runs build/lint/test/security/diff
/skill use review → "review app.py"  ← structured code review when needed
/skill clear                         ← deactivate all skills
```

In [ ]:
# Install dependencies (run once)
!pip install -q boto3 ipywidgets Pillow python-docx pandas openpyxl

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
# Models are imported from sagemaker_agent.py (single source of truth)

import ipywidgets as widgets
from IPython.display import display, HTML
from sagemaker_agent import BEDROCK_MODELS

# Convert BEDROCK_MODELS list-of-tuples to dict for config cell
AVAILABLE_MODELS = dict(BEDROCK_MODELS)

# Temperature options
TEMPERATURE_OPTIONS = {
    "0.0 - Deterministic": 0.0,
    "0.3 - Low creativity": 0.3,
    "0.5 - Balanced": 0.5,
    "0.7 - High creativity": 0.7,
    "1.0 - Maximum creativity": 1.0,
}

# Thinking budget options
THINKING_BUDGET_OPTIONS = {
    "1024 - Minimal": 1024,
    "2048 - Light": 2048,
    "4096 - Standard": 4096,
    "8192 - Extended": 8192,
    "16000 - Maximum": 16000,
}

# Region - Sydney (ap-southeast-2)
REGION = "ap-southeast-2"

# Create configuration widgets
display(HTML("<h3>Agent Configuration</h3>"))

# Use first model as default (matches BEDROCK_MODELS order)
default_model_name = list(AVAILABLE_MODELS.keys())[0]

model_dropdown = widgets.Dropdown(
    options=list(AVAILABLE_MODELS.keys()),
    value=default_model_name,
    description='Model:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='450px')
)

temperature_dropdown = widgets.Dropdown(
    options=list(TEMPERATURE_OPTIONS.keys()),
    value="0.0 - Deterministic",
    description='Temperature:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='350px')
)

thinking_checkbox = widgets.Checkbox(
    value=False,
    description='Enable Extended Thinking (slower, uses more tokens)',
    indent=False,
    style={'description_width': 'auto'}
)

thinking_budget_dropdown = widgets.Dropdown(
    options=list(THINKING_BUDGET_OPTIONS.keys()),
    value="4096 - Standard",
    description='Thinking Budget:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='350px')
)

max_turns_slider = widgets.IntSlider(
    value=60,
    min=5,
    max=100,
    step=5,
    description='Max Turns:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

workspace_input = widgets.Text(
    value='.',
    description='Workspace:',
    placeholder='Directory for file operations',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

mock_toggle = widgets.Checkbox(
    value=False,
    description='Mock Mode (test without API)',
    indent=False,
    style={'description_width': 'auto'}
)

# Display configuration UI
config_box = widgets.VBox([
    model_dropdown,
    widgets.HTML(f"<p style='margin:5px 0;color:#888;'>Region: Sydney ({REGION})</p>"),
    temperature_dropdown,
    thinking_checkbox,
    thinking_budget_dropdown,
    workspace_input,
    max_turns_slider,
    mock_toggle,
], layout=widgets.Layout(padding='10px', border='1px solid #444', margin='10px 0', background='#2d2d2d'))

display(config_box)
display(HTML("<p style='color:#888;font-size:12px;'>Configure settings above, then run the next cell to start.</p>"))

In [ ]:
# ============================================================
# LAUNCH AGENT WITH CONFIGURATION
# ============================================================

from sagemaker_agent import CONFIG, create_chat_ui
from IPython.display import display, HTML

# Security & cost settings (previously in agent_config.json)
CONFIG.aws_bedrock_only = True          # Block ALL AWS services except Bedrock
CONFIG.require_tool_approval = True     # Show Approve/Deny dialog before execution
CONFIG.session_cost_limit = 1.0         # Max $1.00 per session (warns at 80%, stops at 100%)

# Custom slash commands: /review, /explain, /test
CONFIG.custom_commands = {
    "review": {
        "template": (
            "Read $ARGUMENTS and perform a thorough code review using this checklist:\n"
            "\n## 1. Security (CRITICAL)\n"
            "Check: hardcoded secrets, SQL injection, command injection, XSS, path traversal, input validation, auth checks, sensitive data in logs\n"
            "\n## 2. Code Quality (HIGH)\n"
            "Check: functions <50 lines, nesting <4 levels, specific error handling, clear naming, no dead code, no debug statements, DRY, consistent style\n"
            "\n## 3. Performance (MEDIUM)\n"
            "Check: N+1 queries, O(n^2) algorithms, missing caching, unnecessary allocations, lazy processing for large collections\n"
            "\n## 4. Architecture (MEDIUM)\n"
            "Check: separation of concerns, follows existing patterns, no circular deps, config externalized\n"
            "\n## 5. Testing (MEDIUM)\n"
            "Check: critical paths covered, edge cases handled, tests independent, error paths tested\n"
            "\n## Output: Summary, Issues (CRITICAL/HIGH/MEDIUM/LOW with file:line), Positive observations, Suggestions, Rating X/10"
        ),
        "description": "Full code review with 5-category checklist (security, quality, performance, architecture, testing)",
        "agent": "plan"
    },
    "explain": {
        "template": (
            "Read $ARGUMENTS and explain thoroughly:\n"
            "1. PURPOSE: What does this code do? What problem does it solve?\n"
            "2. ARCHITECTURE: How is it structured? What are the main components/classes/functions?\n"
            "3. DATA FLOW: How does data move through the code? Trace a typical request/call.\n"
            "4. KEY DECISIONS: What design patterns are used? Why were they chosen?\n"
            "5. DEPENDENCIES: What does it depend on? What depends on it?\n"
            "6. EDGE CASES: What error handling exists? What could go wrong?\n"
            "Use concrete examples from the actual code. Reference specific line numbers."
        ),
        "description": "Deep code explanation (purpose, architecture, data flow, patterns, dependencies, edge cases)"
    },
    "test": {
        "template": (
            "Read $ARGUMENTS and write comprehensive tests:\n"
            "1. Happy path: normal expected behavior\n"
            "2. Edge cases: empty input, None, boundaries, duplicates, max values\n"
            "3. Error cases: invalid input, missing data, permission errors, timeouts\n"
            "4. Integration: how components work together\n"
            "Use pytest style. Each test: Arrange, Act, Assert. Test names describe what is tested.\n"
            "Target: minimum 8 test functions with good coverage of all branches."
        ),
        "description": "Generate comprehensive test suite (happy path, edge cases, error cases, integration)"
    },
    "verify": {
        "template": (
            "Run 6-phase verification on the current workspace:\n"
            "Phase 1 BUILD: run build/compile if applicable (pip install -e . / npm run build)\n"
            "Phase 2 TYPES: run type checker (mypy/pyright/tsc) if available\n"
            "Phase 3 LINT: run linter (ruff/flake8/eslint) if available\n"
            "Phase 4 TESTS: run test suite (pytest/npm test) with coverage if available\n"
            "Phase 5 SECURITY: grep for hardcoded secrets, .env files, debug statements\n"
            "Phase 6 DIFF: git diff to review all changes\n"
            "\nOutput a VERIFICATION REPORT: each phase PASS/FAIL/SKIP, issues found, ready for PR: YES/NO"
        ),
        "description": "6-phase verification: build, types, lint, tests, security scan, diff review"
    },
    "standards": {
        "template": (
            "Review $ARGUMENTS against these coding standards:\n"
            "NAMING: descriptive vars, verb-noun functions, PascalCase classes, UPPER_SNAKE constants\n"
            "FUNCTIONS: single responsibility, <50 lines, <4 params, <4 nesting, early returns\n"
            "ERRORS: specific exceptions (not bare except), no swallowed errors, user-friendly messages\n"
            "PRINCIPLES: KISS, DRY, YAGNI\n"
            "Fix any violations found. Show before/after for each fix."
        ),
        "description": "Apply coding standards (naming, functions, errors, KISS/DRY/YAGNI)"
    },
}

# Apply configuration from widgets above
CONFIG.model_id = AVAILABLE_MODELS[model_dropdown.value]
CONFIG.region = REGION  # Sydney
CONFIG.workspace = workspace_input.value
CONFIG.max_turns = max_turns_slider.value
CONFIG.mock_mode = mock_toggle.value
CONFIG.temperature = TEMPERATURE_OPTIONS[temperature_dropdown.value]
CONFIG.thinking_enabled = thinking_checkbox.value
CONFIG.thinking_budget = THINKING_BUDGET_OPTIONS[thinking_budget_dropdown.value]

# Display current config
thinking_str = f"Thinking: On (budget: {CONFIG.thinking_budget})" if CONFIG.thinking_enabled else "Thinking: Off"
display(HTML(f"""
<div style="background:#1e3a1e;padding:10px;border-radius:5px;margin:10px 0;color:#d4d4d4;">
<b>Configuration Applied</b><br>
Model: {model_dropdown.value} | Region: Sydney | Temp: {CONFIG.temperature}<br>
{thinking_str} | Mock: {'Yes' if CONFIG.mock_mode else 'No'}
</div>
"""))

# Launch the chat interface
create_chat_ui()

---

## Skills — Detailed Usage Guide

### What Are Skills?

Skills are **text files** (`SKILL.md`) that get injected into the AI's system prompt when activated. They don't add new tools — they change **how the agent thinks and works**. Like giving someone a checklist before they start a job.

### The 5 Available Skills

| Skill | What it does | When to use |
|-------|-------------|-------------|
| `coding-standards` | KISS, DRY, YAGNI rules + naming + function design | **Turn on at session start** — stays active while you code |
| `verify` | Runs bash: build → type check → lint → test → security → git diff | **After you finish coding** — checks everything works |
| `review` | 5-category code review (security/quality/performance/architecture/testing) | **When you want structured feedback** on specific files |
| `report` | Professional report generation (charts-first, structured sections) | When you need a formal document |
| `clara` | ClaRA 5-phase codebase review (discovery → components → readiness → synthesis) | Deep project-level assessment (~$6) |

### Commands

```
/skills                  ← list all available skills
/skill use <name>        ← activate a skill (stays on for ALL messages until cleared)
/skill clear             ← deactivate ALL active skills
/verify                  ← shortcut: auto-activates verify skill + runs it immediately
```

### Typical Session Workflow

```
Step 1: Start session
  /skill use coding-standards          ← turn on standards (stays active)

Step 2: Write code (standards are active — agent follows KISS/DRY/YAGNI)
  "write a function to parse CSV files and return a DataFrame"
  "add error handling for missing columns"
  "refactor the database connection to use connection pooling"

Step 3: Verify your work
  /verify                              ← runs build/lint/test/security/diff

Step 4: Get a code review (optional)
  /skill use review                    ← activate review checklist
  "review the changes I just made"     ← agent follows structured checklist
  /skill clear                         ← deactivate when done

Step 5: Check cost
  /cost                                ← see token usage and spend
```

### Skills Are NOT Linked

Each skill is independent. You CAN stack them (activate multiple), but usually one at a time is best:

```
/skill use coding-standards    ← active
/skill use review              ← NOW BOTH active (standards + review)
/skill clear                   ← clears ALL
```

### Skill vs Slash Command — When to Use Which

| Method | Example | What happens | Persists? |
|--------|---------|-------------|-----------|
| **Slash command** | `/review app.py` | Sends a one-shot prompt with condensed checklist (~15 lines) | No — single message |
| **Skill** | `/skill use review` then "review app.py" | Loads full 83-line checklist into system prompt | Yes — stays active for ALL messages |
| **No skill, no command** | "review app.py" | Agent does its best without a checklist | No |

**Rule of thumb:** Use commands for quick one-off tasks. Use skills when you want consistent behavior across multiple messages.

### Proactive Skill Matching

The agent can **automatically detect** when your request matches a skill:
- Ask "review my code" → agent may auto-load the `review` skill
- Ask "check code quality" → agent may auto-load `coding-standards`

But manual activation with `/skill use <name>` is more reliable.

### Can I Ask V4 to Self-Review Without a Skill?

**Yes.** Just say "review what you just wrote" — the agent will review its own code. The skill just makes the review more structured (won't forget security, performance, etc.). Both work.

### Creating Your Own Skills

Create a folder + `SKILL.md` in the `skills/` directory:

```
skills/
  my-custom-skill/
    SKILL.md
```

Format:
```markdown
---
name: my-custom-skill
description: What this skill does (shown in /skills list)
---

Instructions for the agent here...
```

The agent discovers skills automatically on startup.

---

## Quick Reference

| Action | How |
|--------|-----|
| **Send message** | Type in input box, press Send |
| **Stop agent** | Click Stop button |
| **Check cost** | Type `/cost` (shows token usage, cache savings, session cost) |
| **Code review** | Type `/review filename.py` |
| **Explain code** | Type `/explain filename.py` |
| **Generate tests** | Type `/test filename.py` |
| **Verify project** | Type `/verify` |
| **Apply standards** | Type `/standards filename.py` |
| **Revert file** | Type `/revert filename.py` or `/revert all` |
| **Compact context** | Click Compact button (or auto at 80%) |
| **Clean traces** | Click Clean button (removes audit/snapshots, keeps sessions) |
| **Save/Load** | Save button (auto-saves each message) / Session dropdown + Load |

## Quick Start Examples

| Task | What to type |
|------|-------------|
| Read a file | "Read app.py" |
| Find files | "Find all Python files in this project" |
| Fix a bug | "Read app.py, find the bug, and fix it" |
| Create chart | "Create a bar chart from sales.csv showing revenue by product" |
| Create report | "Create a Word report summarizing the data in results.csv" |
| Run command | "Run git status" |
| Analyze image | "Look at screenshot.png and describe what you see" |
| Search code | "Find where authentication is handled in this codebase" |

## 25+ Tools

| Category | Tools | Approval? |
|----------|-------|-----------|
| **File** | read_file, write_file, edit_file, glob, grep, list_dir | write/edit need approval |
| **Exec** | bash, python_exec | Both need approval |
| **Docs** | create_word, create_excel, create_chart, create_pdf, create_markdown, create_notebook | Need approval |
| **Intelligence** | view_image (vision), semantic_search (code search), web_fetch (URL fetch only) | web_fetch needs approval |
| **Agents** | skill (load checklist), task (spawn sub-agent), ask_user (ask you a question) | task needs approval |
| **State** | todo_write, todo_read | Auto |

## Sub-Agents (6 types)

| Type | What it does | Tools | Max turns |
|------|-------------|-------|-----------|
| **build** | Full development — read, write, execute | All 25+ | 25 |
| **plan** | Architecture analysis (read-only) | 11 | 15 |
| **explore** | Fast file search, codebase navigation | 5 | 10 |
| **verify** | Adversarial testing — tries to BREAK the code | 7 | 15 |
| **review** | Security, quality, performance review | 6 | 10 |
| **general** | General coding tasks | 11 | 15 |

Sub-agents use structured output (Scope, Result, Key files, Issues). The verify agent is explicitly adversarial — it tries to break your code, not just test it.

## Security (16 layers)

- **Bash**: 70 allowed commands only. 75 dangerous patterns blocked.
- **Python**: 63 regex patterns + AST import validation + runtime sandbox.
- **AWS**: `aws_bedrock_only` blocks ALL AWS services except Bedrock.
- **Path traversal**: All file operations enforced to workspace directory.
- **Catastrophic blocks**: `rm -rf /`, `format`, `mkfs` hard-blocked regardless.
- **Pre-edit staleness**: Aborts edit if file changed since last read (prevents silent overwrites).
- **Doom-loop detection**: Warns if agent makes identical repeated tool calls.
- **Cost limit**: Stops at configured session cost cap (warns at 80%).
- **Audit trail**: Every tool call logged with timestamp and integrity hash.

## Prompt Caching

| Feature | Details |
|---------|---------|
| **How** | System prompt + tool schemas cached after turn 1. Subsequent turns ~90% cheaper. |
| **Indicator** | Per-turn: `WRITE 3,228 tok` → `HIT 3,228 tok (saved ~$0.003)` → `INACTIVE` |
| **Cold cache** | After 30 min idle, agent proactively microcompacts before next call |
| **Cache breakage** | After context compact, detects if cache needs rebuilding |
| **Cost display** | `/cost` shows cumulative cache savings in USD |

## Context Management

| Feature | What it does |
|---------|-------------|
| **Microcompact (70%)** | Replace old tool results with markers — buys headroom before expensive compact |
| **Auto-compact (80%)** | Prunes old tool outputs, then LLM summarizes if needed |
| **Post-compact restoration** | Reinjects last 3 recently-read files (up to 32KB) after compact |
| **PTL retry** | Prompt-too-long: trim oldest messages, retry up to 3 times |
| **Diminishing returns** | 3+ consecutive turns with <500 output → advisory warning |

## Memory System (4 types)

| Type | What it stores | Example |
|------|---------------|---------|
| **user** | Your role, preferences, expertise | "Senior Python dev, prefers pytest" |
| **feedback** | Corrections and confirmations | "Don't mock the database in tests" |
| **project** | Ongoing work context, deadlines | "Merge freeze after Thursday" |
| **reference** | Pointers to external resources | "Bugs tracked in Linear project INGEST" |

Stored in `memory.md`. Auto-extracted at session end. Capped at 200 lines / 25KB. The agent reads this at startup and learns about you over time.

## Model Pricing (Bedrock, Sydney region)

| Model | Input / 1M tokens | Output / 1M tokens |
|-------|-------------------|-------------------|
| Claude 3 Haiku | $0.25 | $1.25 |
| Claude 3.5 Haiku | $0.80 | $4.00 |
| **Claude 4.5 Haiku (AU)** | **$1.10** | **$5.50** |
| Claude 4.5 Sonnet (AU) | $3.30 | $16.50 |
| Claude 4.5 Opus | $5.00 | $25.00 |
| Claude 4.6 Opus (AU) | $5.50 | $27.50 |

## Version History

| Version | Key Changes |
|---------|-------------|
| **V4.3.2** | WHEN-not-WHAT tool descriptions, cache-breakage detection, verify agent, bash git safety, absolute paths |
| **V4.3.1** | Prompt engineering: 6 system prompt sections, 7 tool descriptions, sub-agent structured output |
| **V4.3.0** | Diminishing returns, memory 200-line cap, cold-cache microcompact, cache indicator |
| **V4.2.1** | FILE_UNCHANGED_STUB, parallel RO tools, PTL retry |
| **V4.2.0** | Tool result caps, WHAT_NOT_TO_SAVE memory, cold-cache microcompact |
| **V4.1.0** | Cache boundary, catastrophic blocks, 4-type memory |
| **V4.0.0** | Base V4 architecture |

See `USER_GUIDE.md` for full details on every feature.